In [ ]:
# =====================================================
# Install required packages
# =====================================================
!pip install -q transformers accelerate peft bitsandbytes


In [ ]:
# =====================================================
# Imports Required Package
# =====================================================
import torch
import pandas as pd
import numpy as np
import json
import os
import gc
import random
from tqdm.auto import tqdm

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
)
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    confusion_matrix,
    classification_report,
)
from sklearn.model_selection import train_test_split

import warnings
warnings.filterwarnings('ignore')

import logging
logging.getLogger("bitsandbytes").setLevel(logging.ERROR)

print("Imports done")


In [ ]:
# =====================================================
# Mount Google Drive
# =====================================================
from google.colab import drive
drive.mount('/content/drive')

import os
os.chdir('') #EDIT this path to mount to u working folder
print("Working directory:", os.getcwd())


In [ ]:
# =====================================================
# Parameter Configuration
# Labels: 0=Hijazi, 1=Janobi, 2=Najdi, 3=Hasawi).
# =====================================================
DIALECTS = {0: "Hijazi", 1: "Janobi", 2: "Najdi", 3: "Hasawi"}
LABEL2ID = {"Hijazi": 0, "Janobi": 1, "Najdi": 2, "Hasawi": 3}
ID2LABEL = {0: "Hijazi", 1: "Janobi", 2: "Najdi", 3: "Hasawi"}
NUM_LABELS = 4

SORTED_LABEL_IDS = sorted(ID2LABEL)
SORTED_LABEL_NAMES = [ID2LABEL[i] for i in SORTED_LABEL_IDS]

ZEROSHOT_MODEL_IDS = {
    'allam-7b': 'ALLaM-AI/ALLaM-7B-Instruct-preview',
    'mistral-7b': 'mistralai/Mistral-7B-Instruct-v0.3',
    'qwen2.5-7b': 'Qwen/Qwen2.5-7B-Instruct',
}
USE_8BIT = True

MAX_LENGTH = 128

print("Configuration loaded")
print(f"  Dialect mapping: {ID2LABEL}")
print(f"  USE_8BIT: {USE_8BIT}")


In [ ]:
# =====================================================
# Load train/val/test data
# =====================================================
# Pre-split CSVs are included directly in this repository

train_df = pd.read_csv('train_dialectdf.csv')
val_df = pd.read_csv('val_dialectdf.csv')
test_df = pd.read_csv('test_dialectdf.csv')

print(f"Loaded splits: train_dialectdf.csv, val_dialectdf.csv, test_dialectdf.csv")
print(f"  train_df: {len(train_df)}  val_df: {len(val_df)}  test_df: {len(test_df)}")
assert len(train_df) == 3014 and len(val_df) == 377 and len(test_df) == 377, \
    "Split size mismatch -- check that the CSV files match the expected split sizes"


In [ ]:
def extract_dialect_from_response(response_text, dialects=['Hijazi', 'Najdi', 'Hasawi', 'Janobi']):
    """
    Extract dialect label from model response

    Args:
        response_text: Generated text from model
        dialects: List of valid dialect names

    Returns:
        Extracted dialect name or 'Unknown'
    """
    # Normalize response
    response_lower = response_text.lower().strip()

    # Check for each dialect (case-insensitive)
    for dialect in dialects:
        if dialect.lower() in response_lower:
            return dialect

    # Fallback: check first word
    words = response_text.split()
    if words:
        first_word = words[0].strip('.,!?;:')
        for dialect in dialects:
            if first_word.lower() == dialect.lower():
                return dialect

    # If no match, return unknown
    return 'Unknown'

print("extract_dialect_from_response ready")


In [ ]:
# =====================================================
# Zero-shot-v2 / Few-shot-v2 / CoT-v2
# =====================================================

def get_zeroshot_v2_prompt(text):
    """Rules-only, no worked examples."""

    return f"""You are an advanced Arabic NLP classifier specialized in Saudi Arabian regional dialects. Your task is to analyze an isolated word or short phrase and classify it into one of four dialects: Hijazi, Janobi, Najdi, or Hasawi.

### Target Dialect Rules & Indicators

1. **Hijazi (Western Region)**
   - Look for words related to urban idioms, social expressions, or historical Ottoman/Egyptian loanwords.
   - Common markers: Appending "واد" (boy), using "ابو" in expressions, or verbs ending in a fluid "و" suffix (e.g., "اديلو", "سيبك").
   - Baseline vocabulary includes: "كيفك", "دحين", "يا واد", "بكاش", "هرجة", "قوام".

2. **Janobi (Southwestern Mountainous Region)**
   - Look for roots tied to ancient South Arabian vocabulary or unique sentence wrappers.
   - Common markers: Attaching the "ام-" prefix to nouns or verbs (e.g., "امتعطه") or imperative phrases starting with "هب".
   - Baseline vocabulary includes: "ويش كنه", "ارحبوا", "يفدغ", "يلمح", "اندره", "قدهو", "قرعومة", "خطافة", "امعشا".

3. **Najdi (Central Bedouin Region)**
   - Look for traditional central Arabian and Bedouin root structures.
   - Common markers: Frequent usage of "وش" variants, phonetic mutations that turn "ك" into a "تس" sound or "ق" into a "دز" sound, and strong negative pronouns like "مانيب".
   - Baseline vocabulary includes: "وش", "يا رجال", "عساك", "وشلون", "احتريك", "مبطي", "تهقى".

4. **Hasawi (Eastern Oasis/Gulf Region)**
   - Look for words shared with broader Gulf coast dialects and historical trade idioms.
   - Common markers: The distinctive second-person/feminine phonetic shift where "ك" mutates into a "ش" or "ج" sound (e.g., "شبدي" instead of "كبدي", "ابوش" instead of "ابوك").
   - Baseline vocabulary includes: "شلونك", "يهال", "باجر", "جذي", "وايد".

### Classification Constraints
- Provide the output *only* in the requested format.
- If a word could structurally belong to multiple dialects, use the specialized linguistic markers (like prefixes or letter mutations) to break the tie.

### Execution Output Format
Output your final answer strictly using the following keys:
**Input Text:** [The word/phrase being tested]
**Extracted Clues:** [1-2 sentences explaining the linguistic reason or marker identified]
**Predicted Dialect:** [Hijazi, Janobi, Najdi, or Hasawi]

---
**Input Text:** "{text}\""""


def get_fewshot_v2_prompt(text):
    """Rules + 4 hand-crafted worked examples (one per dialect class),
    each demonstrating both the reasoning step and the answer format --
    a manual chain-of-thought demonstration, not sampled from the
    training corpus (unlike the k=5/k=10 sampled-exemplar design this
    replaces)."""

    return f"""You are an advanced Arabic NLP word-level classifier specialized in Saudi Arabian regional dialects. Your task is to analyze an isolated word or short phrase and classify it into one of four dialects: Hijazi, Janobi, Najdi, or Hasawi.

### Target Dialect Schema & Characteristics
* **Hijazi**
  * **Lexical Tendencies:** Uses Ottoman, Egyptian, or diverse Islamic cultural loanwords. Often appends "واد", "ابو", or ends words in a soft "و" (e.g., "اديلو", "سيبك منه").
  * **Core Baseline Identifiers:** "كيفك", "اديلو", "دحين", "يا واد", "بكاش", "هرجة".
* **Janobi**
  * **Lexical Tendencies:** Deeply rooted in ancient South Arabian vocabulary. Frequently uses prefixes like "ويش كنه" or starts verbs/nouns with "ام-" (e.g., "امتعطه") and utilizes distinctive phrasing.
  * **Core Baseline Identifiers:** "ويش كنه", "ارحبوا", "يفدغ", "يلمح", "اندره", "قدهو", "قرعومة", "خطافة", "امعشا".
* **Najdi**
  * **Lexical Tendencies:** Classic central Bedouin root structures. Frequently uses "وش" variants, adds distinct "تس" or "دز" sounds (phonetic mutations of ك/ق), and specific pronouns like "مانيب".
  * **Core Baseline Identifiers:** "وش", "يا رجال", "عساك", "وشلون", "احتريك", "مبطي", "تهقى".
* **Hasawi**
  * **Lexical Tendencies:** Warm Gulf-aligned phonetics. Distinctively shifts the feminine/second-person singular suffixes to a "ش" or "ج" sound (e.g., "شبدي", "ابوش"). Uses historical urban trade terms.
  * **Core Baseline Identifiers:** "شلونك", "يهال", "باجر", "جذي", "وايد".

### Classification Examples (Isolated Words & Short Phrases)
---
**Input Text:** "اشبك يواد"
**Extracted Clues:** Combines an informal Hijazi-style verb form with the regional vocative "يواد".
**Predicted Dialect:** Hijazi
---
**Input Text:** "امتعطه"
**Extracted Clues:** Uses the southwestern "ام-" prefix token combined with a traditional southern verb root.
**Predicted Dialect:** Janobi
---
**Input Text:** "وش دعوى"
**Extracted Clues:** Contains the central Najdi question identifier "وش" paired with a classic regional idiom.
**Predicted Dialect:** Najdi
---
**Input Text:** "بعد شبدي"
**Extracted Clues:** Contains the distinct Eastern/Hasawi phonetic mutation shifting the "ك" to a "ش" ("شبدي" instead of "كبدي").
**Predicted Dialect:** Hasawi

### Instruction
Analyze the isolated word or short phrase below. Extract the primary morphological or lexical clues based on the guidelines above, and output the final classification using the exact same structure.
**Input Text:** "{text}\""""


def get_cot_v2_prompt(text):
    """Rules + a single worked example emphasizing a strict, structured
    3-step reasoning template, rather than per-class example coverage.
    Tests whether forcing explicit, structured reasoning steps helps,
    independent of how many examples are shown."""

    return f"""You are an advanced computational linguist analyzing Saudi Arabian regional dialects (Hijazi, Janobi, Najdi, Hasawi). Analyze the provided Arabic word/phrase, execute a step-by-step reasoning chain in English, and output the final dialect classification.

### Dialect Baseline Rules
- Hijazi: Spoken in Western region. Uses urban idioms, often drops harsh gutturals, or uses fluid "-o" suffix endings (e.g., اديلو).
- Janobi: Spoken in Southwestern region. Rooted in ancient South Arabian phonetics. Often uses the "am-" (ام-) prefix on words.
- Najdi: Spoken in Central region. Classic Bedouin roots. Often mutates "k" to a "ts" sound or "q" to a "dz" sound. Uses "وش".
- Hasawi: Spoken in Eastern/Gulf region. Frequently mutates second-person/feminine "k" sounds to a "sh" (ش) or "j" (ج) sound (e.g., شبدي).

### Example Format
Input Text: "بعد شبدي"
Reasoning Chain:
1. Phonetic Check: The word "شبدي" mutates the standard Arabic letter "ك" into a soft "ش" sound.
2. Morphological Check: The phrase acts as an affectionate, conversational social idiom.
3. Regional Mapping: Shifting "ك -> ش" combined with warm social expressions is an exclusive marker of Hasawi (Eastern Province).
Predicted Dialect: Hasawi

### Task Instruction
Analyze the Input Text below. Write the 3-step Reasoning Chain strictly in English, followed by the Predicted Dialect matching exactly one of the four allowed labels. Do not add any introduction or conclusion text.
Input Text: "{text}"
Reasoning Chain:"""


# =====================================================
# Group A: sampled k-shot prompt, k=5/k=10, sampled once from training data, held fixed across all three models, plain input->label format).
# =====================================================
import random

FEWSHOT_MAX_K = 10

def sample_fewshot_exemplars(train_df, k=FEWSHOT_MAX_K, seed=42):
    """Sample k examples per class from TRAINING data only, once, with
    a fixed seed -- identical exemplar set reused for every model and
    both k values (k=5 uses the first 5 of this same k=10 sample)."""
    rng = random.Random(seed)
    exemplars = {}
    for label_id in SORTED_LABEL_IDS:
        dialect = ID2LABEL[label_id]
        class_texts = train_df[train_df['label'] == label_id]['text'].tolist()
        chosen = rng.sample(class_texts, min(k, len(class_texts)))
        exemplars[dialect] = chosen
    return exemplars

FEWSHOT_EXEMPLARS = sample_fewshot_exemplars(train_df, k=FEWSHOT_MAX_K, seed=42)
print("Sampled few-shot exemplars (Group A, held fixed across all models/conditions):")
for dialect, examples in FEWSHOT_EXEMPLARS.items():
    print(f"  {dialect}: {len(examples)} examples sampled from training data")


def get_sampled_fewshot_prompt(text, k, exemplars=FEWSHOT_EXEMPLARS):
    """Plain input->label few-shot: no forced reasoning, no hand-crafted
    examples -- tests whether example VOLUME from real (sometimes
    messy) training data helps, as a clean, literal test of standard
    k-shot in-context learning."""

    dialect_info = {
        'Hijazi': 'Western Saudi Arabia (Makkah, Madinah, Jeddah)',
        'Najdi': 'Central Saudi Arabia (Riyadh, Al-Qasim)',
        'Hasawi': 'Eastern Saudi Arabia (Al-Hasa, Dammam, Dhahran)',
        'Janobi': 'Southern Saudi Arabia (Aseer, Jazan, Najran)'
    }

    characteristics = {
        'Hijazi': 'Uses "كيفك", "اديلو"',
        'Najdi': 'Uses "وش", "يا رجال", "عساك"',
        'Hasawi': 'Uses "شلونك", "يهال"',
        'Janobi': 'Uses "ويش كنه", "ارحبوا"',
    }

    lines = ["You are an expert in Saudi Arabic dialects.\n",
             "Classify the following Arabic text into ONE of these four Saudi regional dialects:\n"]
    for i, label_id in enumerate(SORTED_LABEL_IDS, 1):
        d = ID2LABEL[label_id]
        lines.append(f"{i}. {d} - {dialect_info[d]}\n   Characteristics: {characteristics[d]}\n")

    lines.append("\nLabeled examples:\n")
    for label_id in SORTED_LABEL_IDS:
        d = ID2LABEL[label_id]
        for ex in exemplars[d][:k]:
            lines.append(f'Text: "{ex}"\nDialect: {d}\n')

    lines.append(f'\nNow classify this new text:\nText: "{text}"\n')
    lines.append("\nInstructions:\n- Analyze the vocabulary, expressions, and linguistic patterns\n"
                  "- Identify the most likely regional dialect\n- Respond with ONLY the dialect name\n")
    lines.append("\nAnswer (one word only - Hijazi, Najdi, Hasawi, or Janobi):")

    return "".join(lines)


def extract_dialect_structured(response_text, dialects=('Hijazi', 'Najdi', 'Hasawi', 'Janobi')):

    for marker in ['predicted dialect', 'final answer']:
        for line in response_text.split('\n'):
            if marker in line.lower():
                for d in dialects:
                    if d.lower() in line.lower():
                        return d
    for d in dialects:
        if d.lower() in response_text.lower():
            return d
    return extract_dialect_from_response(response_text, list(dialects))

print("prompts ready: zero-shot-v2, few-shot-v2, CoT-v2")


In [ ]:
# =====================================================
# Evaluation Function
# =====================================================
def evaluate_fun(model_name, model_id, test_df, condition, max_samples=None):
    """
    condition: one of 'zeroshot_v2', 'fewshot_v2', 'cot_v2'
    """
    PROMPT_FNS = {
        'zeroshot_v2': get_zeroshot_v2_prompt,
        'fewshot_v2': get_fewshot_v2_prompt,
        'cot_v2': get_cot_v2_prompt,
        '5shot_sampled': lambda text: get_sampled_fewshot_prompt(text, k=5),
        '10shot_sampled': lambda text: get_sampled_fewshot_prompt(text, k=10),
    }
    assert condition in PROMPT_FNS, f"Unknown condition: {condition}"
    build_prompt = PROMPT_FNS[condition]

    print(f"\n{'#'*60}\n# Evaluation: {model_name}  [{condition}]\n{'#'*60}\n")

    tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
        tokenizer.pad_token_id = tokenizer.eos_token_id

    print(f"Loading model: {model_id}")
    if USE_8BIT:
        bnb_config = BitsAndBytesConfig(load_in_8bit=True, bnb_8bit_compute_dtype=torch.float16)
        model = AutoModelForCausalLM.from_pretrained(
            model_id, quantization_config=bnb_config, device_map="auto", trust_remote_code=True
        )
    else:
        model = AutoModelForCausalLM.from_pretrained(
            model_id, torch_dtype=torch.float16, device_map="auto", trust_remote_code=True
        )
    model.eval()
    print(f"Model loaded on: {model.device}")

    test_subset = test_df.head(max_samples).copy() if max_samples else test_df.copy()
    print(f"Evaluating on {len(test_subset)} samples...")

    if condition == 'cot_v2':
        max_new_tokens = 120
    elif condition in ('zeroshot_v2', 'fewshot_v2'):
        max_new_tokens = 80
    else:  # 5shot_sampled, 10shot_sampled
        max_new_tokens = 10

    predictions, true_labels, raw_responses, failed_samples = [], [], [], []

    for idx, row in tqdm(test_subset.iterrows(), total=len(test_subset), desc=f"{model_name} [{condition}]"):
        text = row['text']
        true_label = row['label']
        prompt = build_prompt(text)

        inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=1024).to(model.device)

        try:
            with torch.no_grad():
                outputs = model.generate(
                    **inputs,
                    max_new_tokens=max_new_tokens,
                    min_new_tokens=1,
                    do_sample=False,
                    pad_token_id=tokenizer.pad_token_id,
                    eos_token_id=tokenizer.eos_token_id,
                )
            generated_text = tokenizer.decode(
                outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True
            ).strip()
            raw_responses.append(generated_text)

            predicted_dialect = extract_dialect_structured(generated_text)
            predicted_label = LABEL2ID.get(predicted_dialect, -1)

            predictions.append(predicted_label)
            true_labels.append(true_label)

            if predicted_label == -1:
                failed_samples.append({'text': text, 'true_label': ID2LABEL.get(true_label, 'Unknown'), 'response': generated_text})

        except Exception as e:
            print(f"\n⚠ Error on sample {idx}: {e}")
            predictions.append(-1)
            true_labels.append(true_label)
            raw_responses.append(f"ERROR: {str(e)}")

    valid_indices = [i for i, p in enumerate(predictions) if p != -1]
    valid_predictions = [predictions[i] for i in valid_indices]
    valid_true_labels = [true_labels[i] for i in valid_indices]

    if len(valid_predictions) > 0:
        accuracy = accuracy_score(valid_true_labels, valid_predictions)
        precision_w, recall_w, f1_w, _ = precision_recall_fscore_support(valid_true_labels, valid_predictions, average='weighted', zero_division=0)
        precision_m, recall_m, f1_m, _ = precision_recall_fscore_support(valid_true_labels, valid_predictions, average='macro', zero_division=0)
        cm = confusion_matrix(valid_true_labels, valid_predictions, labels=SORTED_LABEL_IDS)
        precision_per_class, recall_per_class, f1_per_class, support = precision_recall_fscore_support(
            valid_true_labels, valid_predictions, average=None, zero_division=0, labels=SORTED_LABEL_IDS
        )
    else:
        print("No valid predictions!")
        accuracy = precision_m = recall_m = f1_m = precision_w = recall_w = f1_w = 0.0
        cm = np.zeros((4, 4))
        precision_per_class = recall_per_class = f1_per_class = np.zeros(4)
        support = np.zeros(4)

    print(f"\nAccuracy:    {accuracy:.4f}")
    print(f"Macro F1:    {f1_m:.4f}   <-- paper's standard metric")
    print(f"Weighted F1: {f1_w:.4f}")
    print(f"Failed/unparsed: {len(failed_samples)} / {len(test_subset)}")

    if len(valid_predictions) > 0:
        print("\n" + classification_report(
            valid_true_labels, valid_predictions,
            labels=SORTED_LABEL_IDS, target_names=SORTED_LABEL_NAMES, digits=4, zero_division=0
        ))

    results = {
        'model_name': model_name, 'model_id': model_id, 'condition': condition,
        'accuracy': float(accuracy),
        'precision': float(precision_m), 'recall': float(recall_m), 'f1': float(f1_m),
        'precision_weighted': float(precision_w), 'recall_weighted': float(recall_w), 'f1_weighted': float(f1_w),
        'true_labels': [int(x) for x in valid_true_labels],
        'pred_labels': [int(x) for x in valid_predictions],
        'per_class_metrics': {
            ID2LABEL[label_id]: {
                'precision': float(precision_per_class[pos]), 'recall': float(recall_per_class[pos]),
                'f1': float(f1_per_class[pos]), 'support': int(support[pos])
            } for pos, label_id in enumerate(SORTED_LABEL_IDS)
        },
        'confusion_matrix': cm.tolist(),
        'confusion_matrix_label_order': SORTED_LABEL_NAMES,
        'n_failed': len(failed_samples),
        'failed_samples': failed_samples[:20],
    }

    json_filename = f'results{model_name}_{condition}.json'
    with open(json_filename, 'w', encoding='utf-8') as f:
        json.dump(results, f, indent=2, ensure_ascii=False)
    print(f"JSON results saved to: {json_filename}")

    del model, tokenizer
    gc.collect()
    torch.cuda.empty_cache()

    return results

print("evaluate_fun() ready")


In [ ]:
# =====================================================
# Verify no prompt example leaks into val/test
# =====================================================
PROMPT_EXAMPLES = {
    'Hijazi': {
        "كيفك": ['zeroshot_v2', 'fewshot_v2'], "دحين": ['zeroshot_v2', 'fewshot_v2'],
        "يا واد": ['zeroshot_v2', 'fewshot_v2'], "بكاش": ['zeroshot_v2', 'fewshot_v2'],
        "هرجة": ['zeroshot_v2', 'fewshot_v2'], "قوام": ['zeroshot_v2', 'fewshot_v2'],
        "اديلو": ['zeroshot_v2', 'fewshot_v2'], "سيبك": ['zeroshot_v2', 'fewshot_v2'],
        "اشبك يواد": ['fewshot_v2'],
    },
    'Janobi': {
        "ويش كنه": ['zeroshot_v2', 'fewshot_v2'], "ارحبوا": ['zeroshot_v2', 'fewshot_v2'],
        "يفدغ": ['zeroshot_v2', 'fewshot_v2'], "يلمح": ['zeroshot_v2', 'fewshot_v2'],
        "اندره": ['zeroshot_v2', 'fewshot_v2'], "امتعطه": ['zeroshot_v2', 'fewshot_v2'],
        "قدهو": ['zeroshot_v2', 'fewshot_v2'], "قرعومة": ['zeroshot_v2', 'fewshot_v2'],
        "خطافة": ['zeroshot_v2', 'fewshot_v2'], "امعشا": ['zeroshot_v2', 'fewshot_v2'],
    },
    'Najdi': {
        "وش": ['zeroshot_v2', 'fewshot_v2', 'cot_v2'], "يا رجال": ['zeroshot_v2', 'fewshot_v2'],
        "عساك": ['zeroshot_v2', 'fewshot_v2'], "وشلون": ['zeroshot_v2', 'fewshot_v2'],
        "احتريك": ['zeroshot_v2', 'fewshot_v2'], "مبطي": ['zeroshot_v2', 'fewshot_v2'],
        "تهقى": ['zeroshot_v2', 'fewshot_v2'], "مانيب": ['zeroshot_v2', 'fewshot_v2'],
        "وش دعوى": ['fewshot_v2'],
    },
    'Hasawi': {
        "شلونك": ['zeroshot_v2', 'fewshot_v2'], "يهال": ['zeroshot_v2', 'fewshot_v2'],
        "باجر": ['zeroshot_v2', 'fewshot_v2'], "جذي": ['zeroshot_v2', 'fewshot_v2'],
        "وايد": ['zeroshot_v2', 'fewshot_v2'], "شبدي": ['zeroshot_v2', 'fewshot_v2', 'cot_v2'],
        "ابوش": ['zeroshot_v2', 'fewshot_v2'], "بعد شبدي": ['fewshot_v2', 'cot_v2'],
    },
}

def check_prompt_leakage(train_df, val_df, test_df, examples=PROMPT_EXAMPLES):
    leaks = []
    checked = 0
    for dialect, words in examples.items():
        for word, prompts in words.items():
            checked += 1
            w = word.strip()
            in_val = (val_df['text'].str.strip() == w).any()
            in_test = (test_df['text'].str.strip() == w).any()
            if in_val or in_test:
                split = 'VAL' if in_val else 'TEST'
                leaks.append({'dialect': dialect, 'word': word, 'split': split, 'affects_prompts': prompts})

    print(f"Checked {checked} example items across all prompts.")
    print("="*70)
    if not leaks:
        print("CLEAN -- no example content leaks into val or test.")
        return leaks

    print(f"⚠ {len(leaks)} LEAK(S) FOUND:")
    for leak in leaks:
        print(f"  '{leak['word']}' ({leak['dialect']}) is in {leak['split']}  -> affects: {', '.join(leak['affects_prompts'])}")
    return leaks

leaks = check_prompt_leakage(train_df, val_df, test_df)
assert not leaks, f"{len(leaks)} leak(s) found -- fix before running the sweep below"


In [ ]:
# =====================================================
# Run Models
# =====================================================
import gc
import os

ZEROSHOT_MODEL_IDS = {
    'mistral-7b': 'mistralai/Mistral-7B-Instruct-v0.3',
    'qwen2.5-7b': 'Qwen/Qwen2.5-7B-Instruct',
    'allam-7b': 'ALLaM-AI/ALLaM-7B-Instruct-preview',
}
CONDITIONS = ['5shot_sampled', '10shot_sampled', 'zeroshot_v2', 'cot_v2', 'fewshot_v2']

results = {}

for model_name, model_id in ZEROSHOT_MODEL_IDS.items():
    results[model_name] = {}
    for condition in CONDITIONS:
        result_path = f"results_{model_name}_{condition}.json"

        if os.path.exists(result_path):
            print(f"Found existing result, skipping: {result_path}")
            with open(result_path) as f:
                results[model_name][condition] = json.load(f)
            continue

        print(f"\n{'*'*70}\nRunning {model_name} [{condition}]\n{'*'*70}")
        try:
            r = evaluate_fun(model_name, model_id, test_df, condition=condition)
            results[model_name][condition] = r
        except Exception as e:
            print(f"{model_name} [{condition}] failed: {e}")
            results[model_name][condition] = None

        gc.collect()
        torch.cuda.empty_cache()

# =====================================================
# Compare against zero-shot
# =====================================================
print(f"\n{'Model':<12}{'Condition':<14}{'Accuracy':<12}{'Macro F1'}")
print("-" * 50)
for model_name in ZEROSHOT_MODEL_IDS:
    for condition in CONDITIONS:
        r = results[model_name].get(condition)
        if r:
            print(f"{model_name:<12}{condition:<14}{r['accuracy']*100:>6.2f}%     {r['f1']*100:>6.2f}%")

with open('results_summary.json', 'w', encoding='utf-8') as f:
    summary = {
        m: {c: (results[m][c]['accuracy'], results[m][c]['f1'])
            for c in results[m] if results[m][c]}
        for m in results
    }
    json.dump(summary, f, indent=2)
print("\n Summary saved to: results_summary.json")
